# Pipeline walkthrough

What the selection pipeline does to a provision, end to end, on committed data.
No network. Deterministic. Run top to bottom.

**Every cell calls library code.** No cell computes a marker hit, a determinacy
rung or a score of its own. If something here needed logic that was not
importable, that was a finding about `simulacria/` and the library was changed
— not the cell. A second implementation in a notebook is worse than no
notebook, because the day it disagrees with the real one you will have
reviewed the wrong system.

Prerequisites: `pip install -e ".[dev]"`, then
`python scripts/build_silver.py` and `python scripts/build_tables.py` for
whichever pool you select below.

In [1]:
import time

import pandas as pd
from IPython.display import HTML, display

from simulacria.selection import highlight
from simulacria.selection.determinacy import classify
from simulacria.selection.inspect import inspect_provision, provenance
from simulacria.selection.markers import ceiling_hits
from simulacria.selection.pools import POOLS, bronze_document, build_provisions, check
from simulacria.selection.shortlist import score_breakdown, shortlist
from simulacria.selection.tables import TABLE_NAMES, query

# Change this one line to re-run the whole notebook against the other pool.
POOL = POOLS["v1"]

# Documents and provisions this walkthrough follows. Both exist in v1 and v2.
DOCUMENT_ID = "sfs-2026-1281"
PROVISION_ID = "sfs-2026-1281:K10P10"

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

print(f"pool {POOL.name}  |  bronze {POOL.bronze_dir}  |  tables {POOL.tables_dir}")

pool v1  |  bronze C:\Users\46762\VSCODE\allegoria\data\bronze\sfs  |  tables C:\Users\46762\VSCODE\allegoria\data\local\tables


## 1. Source → Bronze

The provenance chain: where the bytes came from, and what proves they have not
changed. `provenance()` recomputes the SHA-256 from the file on disk rather
than reading back the one stored beside it — a digest that is only ever read
proves nothing.

`roundtrips_to_bronze` is the stronger check: the `raw_xml` field in the bronze
JSON, re-encoded to UTF-8, must be byte-identical to the source file. That is
what makes the JSON record an honest carrier of the original response.

In [2]:
chain = provenance(DOCUMENT_ID, POOL)
display(
    pd.DataFrame(
        [(k, v) for k, v in chain.items() if k != "bronze_keys"], columns=["field", "value"]
    )
)
print("SHA-256 matches the committed bytes:", chain["sha256_matches"])
print("bronze raw_xml round-trips to those bytes:", chain["roundtrips_to_bronze"])

,field,value
0,document_id,sfs-2026-1281
1,title,Elmarknadslag (2026:1281)
2,version,None
3,source_data_url,https://data.riksdagen.se/dokument/sfs-2026-1281
4,raw_file,C:\Users\46762\VSCODE\allegoria\data\source\sfs\sfs-2026-1281.xml
5,raw_bytes,345765
6,recorded_sha256,b961efcb31d1fdbac18c36ca29b4459a532eb430348a92c7a5c1e1086c0fef63
7,computed_sha256,b961efcb31d1fdbac18c36ca29b4459a532eb430348a92c7a5c1e1086c0fef63
8,sha256_matches,True
9,roundtrips_to_bronze,True


SHA-256 matches the committed bytes: True
bronze raw_xml round-trips to those bytes: True


In [3]:
# The raw XML as committed, and the bronze record built from it.
bronze = bronze_document(DOCUMENT_ID, POOL)
print(str(bronze["raw_xml"])[:600], "\n...")
print("\nbronze fields:", ", ".join(chain["bronze_keys"]))
print(f"\nhtml {len(str(bronze['html'])):,} chars | text {len(str(bronze['text'])):,} chars")

<dokumentstatus><dokument><hangar_id></hangar_id><dok_id>sfs-2026-1281</dok_id><rm>2026</rm><beteckning>2026:1281</beteckning><typ>sfs</typ><subtyp>sfst</subtyp><tempbeteckning></tempbeteckning><organ>Klimat- och näringslivsdepartementet</organ><nummer>1281</nummer><slutnummer>0</slutnummer><datum>2026-06-18 00:00:00</datum><publicerad>2026-06-25 04:34:17</publicerad><systemdatum>2026-06-25 04:34:17</systemdatum><titel>Elmarknadslag (2026:1281)</titel><subtitel></subtitel><status></status><text>/Träder i kraft I:2027-01-01/
1 kap. Inledande bestämmelser

Lagens innehåll

1 § I denna lag f 
...

bronze fields: department, designation, document_id, document_snapshot_id, html, issued_at, payload_size_bytes, published_at, raw_xml, retrieved_at, source_data_url, source_page_url, source_raw_file, source_sha256, source_subtype, source_type, text, title, version

html 175,831 chars | text 121,415 chars


## 2. Bronze → provisions

`build_provisions` parses every committed bronze document. **This is the
slowest cell in the notebook** — it re-parses every document's HTML rather than
reading the built JSONL, so that the contract below is checked against a fresh
parse. It times itself; v2 takes substantially longer than v1.

The count is a contract, not a log line. `check()` returns the list of ways the
build violates its pool's contract: expected document and provision counts,
uniqueness of every derived `provision_id`, and a `source_sha256` on every row.
An empty list is a pass. v2 has no expected counts, so only the structural
checks apply there.

In [4]:
started = time.perf_counter()
provisions, unparsable = build_provisions(POOL.bronze_dir, strict=POOL.strict)
elapsed = time.perf_counter() - started
problems = check(provisions, POOL)

documents = len({str(p["document_id"]) for p in provisions})
print(f"parsed in {elapsed:.1f}s")
print(f"{documents} documents -> {len(provisions):,} provisions")
print(f"expected: {POOL.expected_documents} documents, {POOL.expected_provisions} provisions")
print(f"unparsable, skipped: {len(unparsable)}")
print()
print("CONTRACT: PASS" if not problems else "CONTRACT: FAIL\n  " + "\n  ".join(problems))

parsed in 1.0s
50 documents -> 1,952 provisions
expected: 50 documents, 1952 provisions
unparsable, skipped: 0

CONTRACT: PASS


In [5]:
# One document's provisions.
document_rows = pd.DataFrame(
    [
        {
            "provision_id": p["provision_id"],
            "kind": p["kind"],
            "chapter": p["chapter"],
            "label": p["label"],
            "chars": len(str(p["text"])),
        }
        for p in provisions
        if p["document_id"] == DOCUMENT_ID
    ]
)
print(f"{DOCUMENT_ID}: {len(document_rows)} provisions")
display(document_rows.head(15))

sfs-2026-1281: 320 provisions


,provision_id,kind,chapter,label,chars
0,sfs-2026-1281:K1P1,paragraph,1 kap. Inledande bestämmelser,1 §,116
1,sfs-2026-1281:K1P2,paragraph,1 kap. Inledande bestämmelser,2 §,869
2,sfs-2026-1281:K1P3,paragraph,1 kap. Inledande bestämmelser,3 §,255
3,sfs-2026-1281:K1P4,paragraph,1 kap. Inledande bestämmelser,4 §,63
4,sfs-2026-1281:K1P5,paragraph,1 kap. Inledande bestämmelser,5 §,7894
5,sfs-2026-1281:K1P6,paragraph,1 kap. Inledande bestämmelser,6 §,140
6,sfs-2026-1281:K1P7,paragraph,1 kap. Inledande bestämmelser,7 §,86
7,sfs-2026-1281:K1P8,paragraph,1 kap. Inledande bestämmelser,8 §,169
8,sfs-2026-1281:K2P1,paragraph,2 kap. Systemansvar,1 §,198
9,sfs-2026-1281:K2P2,paragraph,2 kap. Systemansvar,2 §,142


In [6]:
# One provision in full.
chosen = next(p for p in provisions if p["provision_id"] == PROVISION_ID)
print(f"{chosen['provision_id']}  —  {chosen['document_title']}")
print(f"{chosen['chapter']} / {chosen['heading']} / {chosen['label']}")
print(f"{chosen['source_url']}\n")
print(chosen["text"])

sfs-2026-1281:K10P10  —  Elmarknadslag (2026:1281)
10 kap. Mätning av transporterad el / Tvister om kostnader för mätning och för att göra mätresultat  tillgängliga / 10 §
https://data.riksdagen.se/dokument/sfs-2026-1281.html#K10P10

Nätmyndigheten ska ta upp en tvist om vilka kostnader som ska debiteras enligt 6, 7 eller 9 §.

En tvist ska dock inte prövas om ansökan om prövning har kommit in till nätmyndigheten senare än två år efter det att den systemansvariga skickat ett skriftligt ställningstagande till den berörda partens senaste kända adress.


## 3. Markers, inline

The cell that earns the notebook. Every marker hit shown **where it matched**,
coloured by category. Hover any highlight for the full list of markers covering
it — overlaps are real and are kept, because the scoring counts them.

What to look for: whether a marker fired on the words you would have pointed
at. A marker list tells you `om inte` fired; it cannot tell you that the match
ran across a clause boundary and left the conditional it claimed to mark. That
defect was invisible for two sessions and obvious the moment spans were shown
in place.

In [7]:
display(HTML(highlight.legend_html()))
display(HTML(highlight.marker_html(chosen["text"])))

In [8]:
# The same hits as a table, with offsets.
display(pd.DataFrame(highlight.marker_spans(chosen["text"])))

,category,marker,span,start,end
0,duty,ska,ska,15,18
1,qualifier,om,om,35,37
2,duty,ska,ska,58,61
3,exception,ska inte,ska dock inte,105,118
4,duty,ska,ska,105,108
5,exception,dock,dock,109,113
6,qualifier,om,om,126,128
7,qualifier,om,om,137,139
8,qualifier,efter det att,efter det att,200,213


## 4. Determinacy

`DIRECTION.md` grades **the qualifier** on the ladder, not the provision. The
previous specimen sheet got this wrong: it reported provisions as `specific`
because a deadline sat in the duty or the exception, while the qualifier itself
was untestable.

Both figures are shown. `provision_determinacy` still drives the ranking score;
`qualifier_determinacy` is the one to select specimens on. Where they disagree,
the disagreement is the interesting part.

A vague qualifier has one rung left to fall (`vague → absent`); a specific one
has two (`specific → vague → absent`). The founding observation was a specific
qualifier, and the intermediate step is what made it worth studying.

In [9]:
detail = inspect_provision(PROVISION_ID, POOL, provisions=provisions)
print(f"provision-level : {detail['provision_determinacy']}")
print(f"qualifier-level : {detail['qualifier_determinacy']}")
print(
    "\nagree"
    if detail["provision_determinacy"]["state"] == detail["qualifier_determinacy"]["state"]
    else "\nDISAGREE — the checkable bound is not in the qualifier"
)

provision-level : {'state': 'specific', 'specific': ['senare än', 'två år'], 'vague': []}
qualifier-level : {'state': 'specific', 'specific': ['senare än', 'två år'], 'vague': []}

agree


In [10]:
# Which spans were graded, and what each got. `classify` is the same function the
# ladder uses, applied here to one span at a time so the verdict is traceable.
display(
    pd.DataFrame(
        [
            {"marker": label, "start": start, "end": end, "rung": classify(span), "span": span}
            for label, span, start, end in detail["qualifier_spans"]
        ]
    )
)

,marker,start,end,rung,span
0,om,35,80,unmarked,om vilka kostnader som ska debiteras enligt 6
1,om,126,320,specific,om ansökan om prövning har kommit in till nätmyndigheten senare än två år efter det at...
2,om,137,320,specific,om prövning har kommit in till nätmyndigheten senare än två år efter det att den syste...
3,efter det att,200,320,unmarked,efter det att den systemansvariga skickat ett skriftligt ställningstagande till den be...


## 5. Scoring and ranking

The score is deliberately crude and orders a reading list. It is **not** a
measurement: a high rank means "read this first", never "more degraded".

Per category it takes the strongest marker plus a capped bonus for
corroborating ones — summing instead would float a sprawling provision with
eight weak markers above a clean three-part one. Then bonuses for a checkable
qualifier and for twinnable length. Ties break toward shorter text.

In [11]:
breakdown = score_breakdown(chosen["text"])
display(pd.DataFrame([breakdown["components"]]).T.rename(columns={0: "points"}))
print(f"total {breakdown['total']}  |  rank {detail['rank']}  |  {breakdown['char_count']} chars")
print(f"length fits the twinnable band: {breakdown['length_fit']}")
print("\nstrongest marker per category:")
for category, fired in breakdown["category_hits"].items():
    best = max(fired, key=lambda hit: hit[1])
    print(f"  {category:<10} {best[0]:<22} weight {best[1]}   (of {len(fired)} fired)")

,points
duty,2
exception,4
qualifier,2
specific_bonus,4
vague_bonus,0
length_bonus,4


total 16  |  rank 43  |  321 chars
length fits the twinnable band: True

strongest marker per category:
  duty       ska                    weight 2   (of 1 fired)
  exception  dock                   weight 3   (of 2 fired)
  qualifier  om                     weight 1   (of 2 fired)


In [12]:
candidates = shortlist(provisions)
top = pd.DataFrame(
    [
        {
            "rank": i,
            "provision_id": c["provision_id"],
            "score": c["score"],
            "chars": c["char_count"],
            "prov_det": c["determinacy"],
            "qual_det": c["qualifier_determinacy"],
            "exception": ", ".join(c["exception_markers"][:2]),
            "title": str(c["document_title"])[:44],
        }
        for i, c in enumerate(candidates[:15], start=1)
    ]
)
print(f"{len(candidates):,} candidates from {len(provisions):,} provisions")
display(top)

461 candidates from 1,952 provisions


,rank,provision_id,score,chars,prov_det,qual_det,exception,title
0,1,sfs-2026-585:P18,21,732,specific,unmarked,trots,Lag (2026:585) om regeringens godkännande av
1,2,sfs-2026-1054:P8,19,541,specific,unmarked,om inte,Lag (2026:1054) om karens för vissa befattni
2,3,sfs-1977-480:P11,19,548,specific,vague,"dock, om inte",Semesterlag (1977:480)
3,4,sfs-2026-1283:K5P1,19,549,specific,unmarked,"dock, trots",Lag (2026:1283) om elektriska ledningar
4,5,sfs-1982-80:P10,19,555,specific,unmarked,"om inte, i stället",Lag (1982:80) om anställningsskydd
5,6,sfs-1982-80:P20,19,561,specific,unmarked,"om inte, i stället",Lag (1982:80) om anställningsskydd
6,7,sfs-1977-480:P19,19,613,specific,unmarked,"dock, om inte",Semesterlag (1977:480)
7,8,sfs-1982-80:P18,19,675,specific,unmarked,"dock, om inte",Lag (1982:80) om anställningsskydd
8,9,sfs-2026-786:K5P2,19,856,specific,unmarked,i stället,Lag (2026:786) om nordisk verkställighet i b
9,10,sfs-2026-408:K9P10,19,882,specific,unmarked,gäller inte,Vapenlag (2026:408)


## 6. Drill-down

**The cell to use.** Set `PROVISION_ID` at the top of the notebook, or override
it here, and re-run. One library call returns everything the pipeline knows.

In [13]:
target = PROVISION_ID  # <- change me
info = inspect_provision(target, POOL, provisions=provisions)

print(f"{target}  |  pool {info['pool']}  |  candidate: {info['is_candidate']}")
print(f"rank {info['rank']}  score {info['score']}  |  tables {info['tables']}")
print(f"provision determinacy : {info['provision_determinacy']['state']}")
print(f"qualifier determinacy : {info['qualifier_determinacy']['state']}")
if info["ceiling_flags"]:
    print("\nceiling? flags (dock + a bound, UNRESOLVED — hand annotation decides):")
    for window in info["ceiling_flags"]:
        print(f"  {window[:88]}")
print()
display(HTML(highlight.marker_html(info["text"])))
display(pd.DataFrame(info["marker_spans"]))
if info["score_breakdown"]:
    display(pd.DataFrame([info["score_breakdown"]["components"]]).T.rename(columns={0: "points"}))

sfs-2026-1281:K10P10  |  pool v1  |  candidate: True
rank 43  score 16  |  tables {'provisions': 1, 'candidates': 1, 'marker_hits': 9}
provision determinacy : specific
qualifier determinacy : specific



,category,marker,span,start,end
0,duty,ska,ska,15,18
1,qualifier,om,om,35,37
2,duty,ska,ska,58,61
3,exception,ska inte,ska dock inte,105,118
4,duty,ska,ska,105,108
5,exception,dock,dock,109,113
6,qualifier,om,om,126,128
7,qualifier,om,om,137,139
8,qualifier,efter det att,efter det att,200,213


,points
duty,2
exception,4
qualifier,2
specific_bonus,4
vague_bonus,0
length_bonus,4


## 7. Known failure modes

A notebook that shows only the happy path is a demo. These are the audited
defects, displayed so they stay visible rather than living in a review file.

### 7a. `sfs-1982-673:P23` — a false positive, pinned on purpose

`förbjuden` fires on *förbud* in a penalty provision, and the "ett år" that
makes it read `specific` is a **prison term**, not a deadline. It is pinned in
`tests/fixtures/must_surface.yaml` so the marker pair stays covered — the
fixture protects marker coverage, not specimen quality.

In [14]:
fp = inspect_provision("sfs-1982-673:P23", POOL, provisions=provisions)
display(HTML(highlight.marker_html(fp["text"])))
print(f"provision determinacy: {fp['provision_determinacy']}")
print("^ 'ett år' here is a sentence length, read as a checkable qualifier")

provision determinacy: {'state': 'specific', 'specific': ['ett år'], 'vague': []}
^ 'ett år' here is a sentence length, read as a checkable qualifier


### 7b. `sfs-2026-1283:K5P1` — the number-word gap, now closed

Its determinacy read `unmarked` despite a forty-year review period, because the
number words stopped at "trettio". The list now runs to "hundra". The fixture
pinned this provision *to record the gap*, so closing the gap is exactly what
makes `test_fixture_provisions_hit_their_markers` fail — the one deliberately
red test in the suite, awaiting a human decision.

In [15]:
gap = inspect_provision("sfs-2026-1283:K5P1", POOL, provisions=provisions)
print(f"determinacy now: {gap['provision_determinacy']}")
print("fixture pins:    unmarked  <- fails on purpose; do not edit the fixture")
display(HTML(highlight.marker_html(gap["text"])))

determinacy now: {'state': 'specific', 'specific': ['fyrtio år'], 'vague': []}
fixture pins:    unmarked  <- fails on purpose; do not edit the fixture


### 7c. `sfs-2008-567:K1P4` — the corpus's only `såvida inte`

A definitions section, and a poor specimen. Pinned anyway: nothing else in
1,952 provisions exercises that marker, so without it a broken `såvida inte`
would fail silently.

In [16]:
sole = inspect_provision("sfs-2008-567:K1P4", POOL, provisions=provisions)
sole_hits = [h for h in sole["marker_spans"] if h["marker"] == "såvida inte"]
print(f"`såvida inte` hits in this provision: {len(sole_hits)}")
for hit in sole_hits:
    display(
        HTML(highlight.span_html(sole["text"], int(hit["start"]), int(hit["end"]), "såvida inte"))
    )

`såvida inte` hits in this provision: 1


### 7d. A spurious `om inte`, with the clause boundary marked

The open defect. `\bom\s+(?:\w+\s+){0,3}?inte\b` uses a wildcard gap standing
in for "same clause", which a regex cannot express — so the gap steps over the
very tokens that mark the boundary being left. Highlighted in yellow below.

In `sfs-2026-1054:P8` the match is `om att karens inte`: `om att` introduces a
complement clause, not a condition. The provision's *only* exception evidence
is this hit, so it is a candidate purely because of the defect. A fix is scored
in `review/2026-09-11/REVIEW.md` and not shipped.

In [17]:
spurious = inspect_provision("sfs-2026-1054:P8", POOL, provisions=provisions)
om_inte = [h for h in spurious["marker_spans"] if h["marker"] == "om inte"]
exception_markers = [h["marker"] for h in spurious["marker_spans"] if h["category"] == "exception"]
print(f"exception markers firing: {exception_markers}")
for hit in om_inte:
    print(f"\nspan: {hit['span']!r} at offset {hit['start']}")
    display(
        HTML(highlight.clause_boundary_html(spurious["text"], int(hit["start"]), int(hit["end"])))
    )

exception markers firing: ['om inte']

span: 'om att karens inte' at offset 497


### 7e. `sfs-2026-578:P19` — the negative control

Carried on the specimen sheet as a control. On a close reading the second
sentence is a **consequence of breach**, not an exception to the duty. If hand
annotation ever files it as an exception, the slot schema is too permissive.

In [18]:
control = inspect_provision("sfs-2026-578:P19", POOL, provisions=provisions)
display(HTML(highlight.marker_html(control["text"])))
print(
    "markers claim an exception:",
    [h["marker"] for h in control["marker_spans"] if h["category"] == "exception"],
)
print("reading says: consequence of breach, not a carve-out")

markers claim an exception: ['om inte']
reading says: consequence of breach, not a carve-out


### 7f. `dock` doing two different jobs

`ceiling_hits` flags a `dock` followed by a bound. A bound on a granted power
is a **ceiling**, not an exception, and removing one *loosens* where removing
an exception *tightens* — opposite signs from the same marker.

The flag is a screening aid, not a classifier: measured precision on v1 is
about 53%. Resolution is by hand; see
`review/2026-09-11/DIRECTION-ceiling-proposal.md`.

In [19]:
flagged = [
    {"provision_id": c["provision_id"], "window": window[:70]}
    for c in candidates
    for window, bounded in ceiling_hits(str(c["text"]))
    if bounded
]
print(f"{len(flagged)} of {len(candidates):,} candidates carry `dock` + a bound")
display(pd.DataFrame(flagged).head(10))

17 of 461 candidates carry `dock` + a bound


,provision_id,window
0,sfs-1977-480:P11,dock om möjligt minst en månad före ledighetens början. Lag (200
1,sfs-1982-673:P3,"dock under högst en månad, räknat från den dag då avtalet ingick"
2,sfs-1977-480:P13,dock endast om arbetstagaren underrättar arbetsgivaren senast tv
3,sfs-2026-772:K1P4:occurrence:2,dock ha gällt i minst ett år.
4,sfs-2026-1283:K2P6,dock under högst tre år.
5,sfs-2026-786:K4P3,dock längst under 96 timmar.
6,sfs-2026-1011:K8P28,"dock minst tre månader, har kreditgivaren trots vad som anges i"
7,sfs-1982-80:P6c,dock senast den sjunde kalenderdagen efter det att arbetstagaren
8,sfs-2026-786:K10P20,dock längst under 48 timmar.
9,sfs-2026-786:K4P4,dock längst under 96 timmar.


## 8. Tables

The Parquet layer the shortlist is queried through. `marker_hits` covers **all**
provisions, not only candidates — that is what keeps near-misses and
never-firing markers answerable.

The queries below are from `docs/queries.md`, run here so the query path is
exercised rather than described.

In [20]:
for name in TABLE_NAMES:
    columns, rows = query(POOL.tables_dir, f'select count(*) from "{name}"')
    schema_cols, schema_rows = query(POOL.tables_dir, f'describe "{name}"')
    print(f"\n{name}: {rows[0][0]:,} rows")
    display(pd.DataFrame(schema_rows, columns=schema_cols)[["column_name", "column_type"]].T)


provisions: 1,952 rows


,0,1,2,3,4,5,6,7,8,9,10,11,12
column_name,provision_id,document_id,document_title,document_version,kind,chapter,label,heading,order,text,char_count,source_url,source_sha256
column_type,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,INTEGER,VARCHAR,INTEGER,VARCHAR,VARCHAR



candidates: 461 rows


,0,1,2,3,4,5,6,7,8,9,10,11
column_name,provision_id,rank,score,has_duty,has_exception,has_qualifier,determinacy,qualifier_determinacy,duty_marker,exception_marker,qualifier_marker,char_count
column_type,VARCHAR,INTEGER,DOUBLE,BOOLEAN,BOOLEAN,BOOLEAN,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,INTEGER



marker_hits: 8,983 rows


,0,1,2,3,4
column_name,provision_id,category,marker,matched_span,char_offset
column_type,VARCHAR,VARCHAR,VARCHAR,VARCHAR,INTEGER



markers: 28 rows


,0,1,2,3
column_name,category,marker,weight,pattern
column_type,VARCHAR,VARCHAR,INTEGER,VARCHAR


In [21]:
# docs/queries.md #2 — determinacy across candidates, both figures side by side.
columns, rows = query(
    POOL.tables_dir,
    """
    select determinacy, qualifier_determinacy, count(*) as candidates
    from candidates group by 1, 2 order by candidates desc
    """,
)
display(pd.DataFrame(rows, columns=columns))

,determinacy,qualifier_determinacy,candidates
0,unmarked,unmarked,298
1,specific,unmarked,62
2,specific,specific,52
3,vague,vague,23
4,vague,unmarked,21
5,specific,vague,5


In [22]:
# docs/queries.md #3 — marker frequency, and which markers never fire. The left
# join is the point: a marker that never matches has no rows to count.
columns, rows = query(
    POOL.tables_dir,
    """
    select m.category, m.marker, m.weight,
           count(h.provision_id) as hits,
           count(distinct h.provision_id) as provisions
    from markers m
    left join marker_hits h on h.category = m.category and h.marker = m.marker
    group by 1, 2, 3 order by hits asc
    """,
)
display(pd.DataFrame(rows, columns=columns))

,category,marker,weight,hits,provisions
0,exception,såvida inte,3,1,1
1,duty,måste,2,4,4
2,exception,utan hinder av,3,6,5
3,duty,bör,1,8,7
4,exception,utom,3,15,14
5,exception,behöver inte,2,17,14
6,exception,trots,3,20,20
7,qualifier,under förutsättning att,4,29,28
8,duty,förbjuden,2,38,30
9,duty,är skyldig,2,40,40


In [23]:
# docs/queries.md #5 — near-misses: duty and exception, but no qualifier. These
# are not candidates, which is why marker_hits must span every provision.
columns, rows = query(
    POOL.tables_dir,
    """
    with parts as (
      select provision_id,
             count(*) filter (where category = 'duty')      > 0 as has_duty,
             count(*) filter (where category = 'exception') > 0 as has_exception,
             count(*) filter (where category = 'qualifier') > 0 as has_qualifier
      from marker_hits group by provision_id
    )
    select p.provision_id, p.label, p.char_count, p.document_title
    from parts join provisions p using (provision_id)
    where has_duty and has_exception and not has_qualifier
    order by p.char_count limit 10
    """,
)
display(pd.DataFrame(rows, columns=columns))

,provision_id,label,char_count,document_title
0,sfs-2026-1281:K9P8,8 §,89,Elmarknadslag (2026:1281)
1,sfs-2026-1281:K20P3,3 §,133,Elmarknadslag (2026:1281)
2,sfs-2026-1283:K12P3,3 §,154,Lag (2026:1283) om elektriska ledningar
3,sfs-2026-966:K3P3,3 §,178,Lag (2026:966) om utsedda verksamhetsställen och rättsliga ombud för inhämtning av ele...
4,sfs-2026-1281:K11P2,2 §,189,Elmarknadslag (2026:1281)
5,sfs-2026-1283:K12P2:occurrence:2,2 §,190,Lag (2026:1283) om elektriska ledningar
6,sfs-2026-1281:K4P4,4 §,199,Elmarknadslag (2026:1281)
7,sfs-2026-1281:K12P26,26 §,204,Elmarknadslag (2026:1281)
8,sfs-1982-80:overgang:2006-439,SFS 2006:439,206,Lag (1982:80) om anställningsskydd
9,sfs-1977-480:overgang:1994-1688,SFS 1994:1688,207,Semesterlag (1977:480)


## What this notebook does not show

- **Which part a qualifier attaches to.** It decides the sign of a removal and
  `DIRECTION.md` assigns it to hand annotation. Nothing here infers it.
- **Whether a `ceiling?` flag is really a ceiling.** Same reason.
- **`direction` itself.** That is `simulacria.measurement`, empty until Phase 2,
  and it may not import anything shown above.